# B0 no Colab — sweep de resolução (Task 2)

Roda o experimento **B0** num T4 gratuito do Colab: treina o Qwen2.5-VL-3B com QLoRA
e avalia, variando o orçamento de resolução.

**Antes de começar:** `Ambiente de execução > Alterar tipo de ambiente > T4 GPU`.

---
### Por que um orçamento por sessão

O Colab gratuito desconecta por inatividade e tem limite de sessão. O sweep completo
(4 orçamentos) não cabe numa sessão só. Este notebook roda **um orçamento por vez** e
salva tudo no Drive, então você roda 100, volta depois e roda 334, e assim por diante.

Se o adaptador daquele orçamento já existir no Drive, o treino é pulado automaticamente.


## 1. Conferir a GPU

O T4 é Turing (compute 7.5) e **não tem bf16 nativo**. O código já detecta isso e cai
pra fp16 sozinho — essa célula só mostra o que você recebeu.


In [ ]:
!nvidia-smi

import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU alocada:', out or 'NENHUMA')
assert out, 'Sem GPU. Ambiente de execucao > Alterar tipo de ambiente > T4 GPU'


## 2. Montar o Drive

Tudo que precisa sobreviver à desconexão vai pra cá: dataset, adaptadores e relatórios.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/sci-image-markdown-b0'
os.makedirs(WORK, exist_ok=True)
print('Persistencia em:', WORK)


## 3. Clonar o repositório

⚠️ A branch `b0-resolucao-simetrica` precisa estar **publicada** antes disso.
Se ela ainda só existe na sua máquina, rode lá: `git push origin b0-resolucao-simetrica`.

Ajuste `REPO_URL` para o fork/remote onde a branch está.


In [ ]:
REPO_URL = 'https://github.com/lucasdocunha/sci-image-markdown.git'  # ajuste se usar fork
BRANCH = 'b0-resolucao-simetrica'

import os
if not os.path.exists('/content/repo'):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} /content/repo
else:
    !cd /content/repo && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull

%cd /content/repo
!git log --oneline -1


## 4. Instalar dependências

O Colab já traz torch com CUDA, então instalamos só o resto. Leva uns minutos.


In [ ]:
!pip install -q transformers peft accelerate bitsandbytes qwen-vl-utils rouge-score nltk python-Levenshtein tabulate click rich

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0))
cc = torch.cuda.get_device_capability()
print(f'compute capability: {cc[0]}.{cc[1]}  ->  bf16 nativo:', cc[0] >= 8)


## 5. Baixar o dataset

~1200 figuras. Fica no Drive, então só baixa na primeira vez.


In [ ]:
import os
DATA = f'{WORK}/data'

if os.path.exists(f'{DATA}/processed/train.jsonl'):
    print('Dataset ja esta no Drive, pulando o download.')
else:
    !python prepare_data.py --download-hf --raw-dir {DATA}/raw --processed-dir {DATA}/processed --num-workers 16

!wc -l {DATA}/processed/*.jsonl


## 6. Escolher o orçamento desta sessão

| orçamento | ~px | figuras não reduzidas | tempo estimado no T4 |
|---:|---:|---:|---|
| **100** | 280px | 1% | ~1-2 h |
| **334** | 511px | 19% | ~3-4 h |
| **752** | 767px | 77% | ~6-8 h ⚠️ |
| **1337** | 1023px | 96% | pode dar OOM no T4 ⚠️ |

Os tempos são estimativas grosseiras (treino + avaliação das 373 figuras de teste) e
variam bastante. **Comece pelo 100** — é o baseline e o mais rápido; só assim os outros
têm contra o que ser comparados.

Se o 752 ou o 1337 estourar a memória, isso **é um resultado**: anote e siga. É
exatamente o teto de VRAM que o sweep existe pra encontrar.


In [ ]:
BUDGET = 100   # <<< mude aqui a cada sessao: 100 -> 334 -> 752 -> 1337

import os
RUN = f'{WORK}/b0_sweep/res_{BUDGET}'
os.makedirs(RUN, exist_ok=True)
print(f'orcamento: {BUDGET} tokens (~{int((BUDGET*784)**0.5)}px)')
print('saida em:', RUN)


## 7. Treinar

Época fixa, sem seleção de checkpoint — é o que mantém os orçamentos comparáveis
entre si (o conjunto de validação é pequeno demais pra escolher checkpoint sem ruído).

Deixe a aba aberta: o Colab desconecta se achar que você abandonou a sessão.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

ADAPTER = f'{RUN}/checkpoints/final_adapter'
if os.path.exists(ADAPTER):
    print('Adaptador ja existe para este orcamento, pulando o treino.')
else:
    !python train.py --config configs/default.yaml \
        -o data.max_visual_tokens={BUDGET} \
        -o data.train_file={DATA}/processed/train.jsonl \
        -o data.val_file={DATA}/processed/val.jsonl \
        -o data.image_folder={DATA}/processed \
        -o training.output_dir={RUN}/checkpoints \
        -o experiment_name=b0_res_{BUDGET} 2>&1 | tee {RUN}/train.log


## 8. Avaliar

**No mesmo orçamento do treino** — essa simetria é o ponto do B0 inteiro.


In [ ]:
!python evaluate.py --config configs/default.yaml \
    --test-file {DATA}/processed/test.jsonl \
    --adapter-path {RUN}/checkpoints/final_adapter \
    --output-report {RUN}/report_finetuned.json \
    --save-predictions {RUN}/predictions_finetuned.jsonl \
    -o data.max_visual_tokens={BUDGET} \
    -o data.image_folder={DATA}/processed 2>&1 | tee {RUN}/eval.log


## 9. Comparar tudo que já rodou

Lê todos os relatórios que existem no Drive e monta a tabela. Rode depois de cada
sessão pra ver o sweep crescendo.


In [ ]:
!python scripts/summarize_sweep.py {WORK}/b0_sweep


---
## Se der `CUDA out of memory`

Em ordem, antes de desistir do orçamento:

1. `Ambiente de execução > Reiniciar sessão` e rode de novo (memória fragmentada).
2. Suba o acúmulo de gradiente em vez do batch: `-o training.gradient_accumulation_steps=16`
3. Se ainda assim estourar, **anote e siga**. O teto de VRAM é um resultado do
   experimento, não uma falha sua — e é o que justifica pedir a GPU maior do Lucas.
